# VocDenoiser — Colab driver

Thin driver that calls into the `vocdenoiser` package. Three stages:
1. **SNR filtering** — score clips, pick a clean training subset (CPU, numpy-only).
2. **Train** the β-VAE denoiser (GPU).
3. **Architecture search** — autoresearch-style loop over model families (GPU).

Runtime → Change runtime type → **GPU (A100/L4)** for stages 2–3.

In [ ]:
# Install (SNR-only: `pip install -e .`; with ML stack: `.[ml]`)
!git clone https://github.com/kalleknast/VocDenoiser.git
%cd VocDenoiser
!pip install -q -e '.[ml]'

In [ ]:
# Get the clean calls AND the real colony-noise onto Colab-LOCAL disk (never train off Drive).
from google.colab import drive
drive.mount('/content/drive')
import os
DRIVE_DIR = '/content/drive/MyDrive/MarmosetData'   # where the three zips were uploaded
for z in ['Vocalizations', 'Noise', 'Cigarra']:
    print(f'Unzipping {z}.zip ...')
    !unzip -q -o '{DRIVE_DIR}/{z}.zip' -d /content
# Verify all three extracted (expect ~50000 / 30 / 136):
!echo "Vocalizations:" $(ls /content/Vocalizations 2>/dev/null | wc -l) \
      "| Noise:" $(ls /content/Noise 2>/dev/null | wc -l) \
      "| Cigarra:" $(ls /content/Cigarra 2>/dev/null | wc -l)
os.environ['VOCDENOISER_DATA_ROOT'] = '/content/Vocalizations'   # 50k clean phee calls
# Real recorded backgrounds for the training noise mix: /content/Noise, /content/Cigarra

## 1. SNR filtering — select a clean training subset

In [ ]:
# Score every clip, then inspect the distribution before choosing a cutoff.
!python -m vocdenoiser.cli snr scan $VOCDENOISER_DATA_ROOT --out artifacts/snr_scores.csv --workers 8
!python -m vocdenoiser.cli snr report artifacts/snr_scores.csv --out-dir reports
# (optional) confirm the SNR metric is call-agnostic on THIS data, using the real colony noise.
!python -m vocdenoiser.cli snr validate artifacts/snr_scores.csv \
    --src-dir $VOCDENOISER_DATA_ROOT --noise-dir /content/Noise --noise-dir /content/Cigarra --out-dir reports
from IPython.display import Markdown
Markdown(open('reports/snr_report.md').read())

In [ ]:
# Pick a threshold from the report, then select the clean subset.
# Decided recipe for this data: primary snr_db cutoff at the Otsu crossover
# (~19.82 dB on the current scan -> ~19.5k clips), plus a secondary broadband
# floor. The broadband floor is a cheap safeguard against hissy narrowband/tonal
# calls that the primary snr_db under-penalizes; at the 19.82 dB cutoff it only
# trims ~0.1% (the primary cutoff already excludes most of them), but it matters
# if you lower --threshold to recover more clips. Re-read reports/ for THIS scan.
!python -m vocdenoiser.cli snr select artifacts/snr_scores.csv \
    --src-dir $VOCDENOISER_DATA_ROOT --out artifacts/clean_manifest.csv \
    --threshold 19.82 --broadband-floor 15.0 --link-dir /content/clean_subset

## 2. Train the β-VAE denoiser

In [ ]:
# Train on the clean subset (noisy→clean pairs synthesised on the fly). The noise mix is the
# synthetic recipe (40% pink / 50% babble / 10% transients) BLENDED with the real recorded
# backgrounds (--noise-dirs). Tune the blend with --real-noise-weight (0 = synthetic only,
# 1 = real only, default 0.5).
# --num-workers: the augmentation is CPU-bound (phase-vocoder pitch/stretch), so parallel
# workers keep the GPU fed. 4 is a good start on an L4 (raise toward the vCPU count if the
# GPU is still starved; drop to 2 if RAM is tight).
!python -m vocdenoiser.denoise.train \
    --data-root /content/clean_subset --max-epochs 100 --batch-size 32 --beta 4.0 \
    --noise-dirs /content/Noise /content/Cigarra --num-workers 4

In [ ]:
# Evaluate: UMAP of the 16-dim latents + RandomForest identity-preservation proxy.
# data/Vocalizations filenames are bare numeric call IDs with NO identity in the
# path, so --label-from parent/stem would give a meaningless RF score. Pass an
# id->individual mapping CSV instead (columns: id,identity). Without it the UMAP
# still renders and the RF proxy is skipped.
!python -m vocdenoiser.denoise.eval \
    --data-root /content/clean_subset --ckpt checkpoints/last.ckpt \
    --labels-csv individuals.csv

## 2b. Cross-dataset caller-identity benchmark (InfantMarmosetsVox)

An external check of "does compression preserve individual identity?" on a dataset with
ground-truth **caller identity** ([InfantMarmosetsVox](https://zenodo.org/records/10130104),
CC-BY-4.0; 10 infant marmosets, 11 call types). The loader downloads the audio, cuts each
annotated vocalization, upsamples 44.1→96 kHz to match the model, **scores call quality**
(SNR + clipping + level) and drops noisy / near-silent clips, then writes an `id,identity`
(=caller) CSV.

Read the RandomForest accuracy as a **cross-dataset** number — a different colony (infant
twins), and an empty 22–48 kHz band after upsampling — not comparable to an in-domain split.
Needs the checkpoint trained in section 2.

In [ ]:
# Cross-dataset caller-identity benchmark. Runs on Colab LOCAL disk (fast, ephemeral);
# needs the checkpoint from section 2.
IMV_ROOT = '/content/InfantMarmosetsVox'   # NOT on Drive — re-downloadable external data

# Download (~21 GB) + extract + cut per-call clips at 96 kHz + SCORE CALL QUALITY.
# External colony/rig, so quality varies: --min-snr / --min-peak-dbfs drop noisy or
# near-silent clips; the printed summary shows the distribution. identity = caller.
!python -m vocdenoiser.datasets.infantmarmosetsvox \
    --download --imv-root $IMV_ROOT --target-sr 96000 \
    --min-snr 6 --min-peak-dbfs -40

# (optional) reclaim ~21 GB once clips are cut — the 10-min recordings aren't needed after:
# !rm -rf $IMV_ROOT/data

# Identity-preservation proxy on the compressed latents of a DIFFERENT colony.
!python -m vocdenoiser.denoise.eval \
    --data-root $IMV_ROOT/clips --ckpt checkpoints/last.ckpt \
    --labels-csv $IMV_ROOT/imv_labels.csv \
    --out-png umap_imv.png --out-latents latents_imv.npy

from IPython.display import Image
Image('umap_imv.png')

## 3. Architecture search (autoresearch-style)

Greedy hill-climb over a top-K frontier under a fixed compute budget; each
candidate trained for `--max-steps` and scored by spectrogram SI-SDR. The JSONL
ledger is the resumable state.

In [ ]:
# Dry-run the loop with NO GPU (synthetic landscape) to see the mechanics:
!python -m vocdenoiser.cli search run --harness mock --iters 30 --ledger artifacts/mock.jsonl
!python -m vocdenoiser.cli search report --ledger artifacts/mock.jsonl

In [ ]:
# Real search (GPU): trains each candidate for max-steps on the clean subset.
!python -m vocdenoiser.cli search run --harness torch \
    --data-root /content/clean_subset --iters 60 --max-steps 400 \
    --seeds 0 1 --ledger artifacts/search_ledger.jsonl
!python -m vocdenoiser.cli search report --ledger artifacts/search_ledger.jsonl